# Traffic Dataset - Data Processing

Processing pipeline for the Madrid traffic dataset (all available months).

## Specification

1. **Load dataset and filter**: Read all CSV files from `dataset/`, filter to urban (`C30`) and interurban (`M30`) measurement points, and cache to parquet for fast subsequent loads. Inspect dtypes, summary statistics, and null counts.
2. **Inspect errors and missing data**: Examine error field distribution (`N` vs `NaN`), `vmed` distribution (including negative values), and null counts.
3. **Data processing: 15-min to hourly aggregation**:
   1. **Data cleaning**:
      - Invalidate rows with `error='S'` or `error=NaN` (set all numeric fields to NaN).
      - Replace negative values with NaN (negative = missing data per source documentation).
      - Flag each 15-min row as valid/invalid based on `intensidad` being non-NaN.
   2. **Hourly aggregation** (per `id`, per hour):
      - **4/4 valid quarters**: aggregate normally.
      - **3/4 valid quarters**: aggregate + flag `hora_baja_confianza=1`.
      - **<=2/4 valid quarters**: all values set to NaN + flag `hora_incompleta=1`.
      - Aggregation functions: `intensidad` (mean), `ocupacion` (mean), `vmed` (mean), `error` (worst: S > E > N), `periodo_integracion` (sum).
      - `intensidad` remains in its original unit (vehicles/hour); the mean of valid quarters estimates the hourly rate.
      - Quality flags added:
        - `n_validos_15m`: number of valid 15-min quarters in the hour (0–4). A quarter is valid if `intensidad` is not NaN after data cleaning.
        - `n_error_E`: number of quarters in the hour where the source reported suboptimal sample quality (`error='E'`).
        - `n_error_S`: number of quarters in the hour where the source reported a fully erroneous sample (`error='S'`).
        - `n_error_NaN`: number of quarters in the hour where the error field is NaN (broken/unresponsive sensors). These are the primary cause of incomplete hours.
        - `hora_incompleta` (0/1): set to 1 when fewer than 3 valid quarters are available, meaning the hour lacks sufficient data to produce a reliable aggregate. All numeric fields are set to NaN for these hours.
        - `hora_baja_confianza` (0/1): set to 1 when exactly 3 of 4 quarters are valid. The aggregate is computed but may be less representative than a full-coverage hour.
   3. **Final hourly dataset**: Verify shape, dtypes, null counts, and spot-check sample measurement points. Analyze per-ID row counts and coverage thresholds to inform the minimum coverage filter.
4. **Filter by minimum coverage**: Keep only measurement points with at least 25,000 hourly records, ensuring sufficient temporal coverage for analysis.
5. **Reindex to complete hourly range**: Reindex each sensor to the full hourly range (2023-01-01 00:00 to 2026-02-28 23:00 = 27,720 hours). Missing hours become explicit NaN rows so that subsequent interpolation correctly measures actual time gaps instead of row positions. New rows are flagged as `hora_incompleta=1` with `n_validos_15m=0`. After this step every sensor has exactly 27,720 rows.
6. **Imputation**:
   - Imputation flags and loss weight are created **before** any values are filled:
     - `intensidad_is_imputed`, `ocupacion_is_imputed`, `vmed_is_imputed`: `'True'` for NaN rows, `'False'` for real data.
     - `peso_loss`: `1.0` for real observations, `0.0` for imputed values. Used to exclude imputed rows from the loss function during model training.
   - Imputation proceeds in four stages (per `id`), applied to `intensidad`, `ocupacion`, and `vmed`:
     1. **Linear interpolation** (`limit=3`): fills gaps of up to 3 consecutive NaN hours by linearly interpolating between surrounding valid values.
     2. **Strict seasonal mean** (month + day-of-week + hour): fills remaining NaN values with the sensor's historical mean for the same month, day-of-week, and hour-of-day.
     3. **Relaxed seasonal mean** (day-of-week + hour): fallback for hours where the strict grouping has no data.
     4. **Hour-of-day mean**: final fallback using only the hour-of-day.
7. **Feature enrichment**: Add calendar features derived from `fecha`:
   - `dia_semana` (int, 0–6): day of week (0=Monday, 6=Sunday).
   - `hora_dia` (int, 0–23): hour of the day.
   - `mes` (int, 1–12): month of the year.
   - `festivo` (0/1): whether the date is a public holiday in Madrid. Holidays are sourced from the `holidays` library (national + Comunidad de Madrid, using `observed=False` to flag the actual calendar dates) plus manually added city-level holidays: San Isidro (May 15) and La Almudena (Nov 9).
   - `vispera_festivo` (0/1): eve of a holiday — set to 1 when the current day is not a holiday (`festivo=0`) but the following day is (`festivo=1`).
8. **Export**: Write the final dataset to `dataset/processed/traffic.csv`.

---

> **Note — Missing hours in the source dataset**
>
> The expected hourly time series from 2023-01-01 00:00 to 2026-02-28 23:00 spans 27,720 hours. Before the reindex step, the best-covered sensors had only 27,662 hours (58 missing). These break down into two categories:
>
> **51 hours globally absent** — no C30 or M30 sensor has any record:
>
> | Dates | Missing hours | Count |
> |---|---|---|
> | 2025-01-30 | 00:00 – 01:00 | 2 h |
> | 2025-04-28 to 2025-04-30 | 2025-04-28 16:00 – 2025-04-30 13:00 | 36 h |
> | 2025-06-16 | 18:00 – 20:00 | 3 h |
> | 2025-11-12 | 15:00 – 19:00 | 5 h |
> | 2025-12-23 | 23:00 | 1 h |
> | **Subtotal** | | **51 h** |
>
> **7 hours absent across all M30 sensors** — present for ~120 C30 sensors but missing for all M30 sensors (which have the highest overall coverage):
>
> | Dates | Missing hours | Count |
> |---|---|---|
> | 2023-03-13 | 12:00 | 1 h |
> | 2023-08-15 | 03:00 – 05:00 | 3 h |
> | 2024-07-16 | 13:00 – 14:00 | 2 h |
> | 2025-04-29 | 03:00 | 1 h |
> | **Subtotal** | | **7 h** |
>
> **Total: 51 + 7 = 58 missing hours** for the best-covered (M30) sensors. No single sensor in the dataset covers all 27,669 non-globally-missing hours. These gaps originate from the Madrid open data source (no 15-min records were published for these periods). After the reindex step they become explicit NaN rows and are filled by the seasonal imputation fallbacks. They are marked with `peso_loss=0.0` and excluded from the loss function during training.

In [1]:
%pip install pandas numpy pyarrow
import pandas as pd
import numpy as np
from pathlib import Path
import glob

  Using cached pyarrow-23.0.1-cp312-cp312-macosx_12_0_arm64.whl.metadata (3.1 kB)
Using cached pyarrow-23.0.1-cp312-cp312-macosx_12_0_arm64.whl (34.2 MB)
Note: you may need to restart the kernel to use updated packages.


## 1. Load Dataset and Filter

Load from parquet cache if available (fast), otherwise read raw CSVs, filter by measurement type, and cache.

In [2]:
PARQUET_CACHE = 'dataset/filtered_c30_m30.parquet'

if Path(PARQUET_CACHE).exists():
    print('Loading from parquet cache...')
    df = pd.read_parquet(PARQUET_CACHE)
    print(f'Shape: {df.shape}')
else:
    csv_files = sorted(glob.glob('dataset/*.csv'))
    print(f'Found {len(csv_files)} CSV files — loading and filtering...')
    df = pd.concat(
        [pd.read_csv(f, sep=';', quotechar='"') for f in csv_files],
        ignore_index=True
    )
    print(f'Raw shape: {df.shape}')
    df = df[df['tipo_elem'].isin(['C30', 'M30'])]
    print(f'After C30/M30 filter: {df.shape}')
    df.to_parquet(PARQUET_CACHE, index=False)
    print(f'Cached to {PARQUET_CACHE}')

df = df.drop(columns=['carga'])
df['tipo_elem'].value_counts()

display(df[df['id'] == 6900])
display(df[df['id'] == 6899])

Loading from parquet cache...
Shape: (43409606, 9)


,id,fecha,tipo_elem,intensidad,ocupacion,vmed,error,periodo_integracion
978170,6900,2023-01-01 00:00:00,C30,420,4.0,54.0,N,5
978171,6900,2023-01-01 00:15:00,C30,420,4.0,54.0,N,5
978172,6900,2023-01-01 00:30:00,C30,0,0.0,0.0,N,5
978173,6900,2023-01-01 00:45:00,C30,0,0.0,0.0,N,5
978174,6900,2023-01-01 01:00:00,C30,0,0.0,0.0,N,5
...,...,...,...,...,...,...,...,...
43126780,6900,2025-12-31 22:45:00,C30,0,0.0,0.0,N,5
43126781,6900,2025-12-31 23:00:00,C30,0,0.0,0.0,N,5
43126782,6900,2025-12-31 23:15:00,C30,0,0.0,0.0,N,5
43126783,6900,2025-12-31 23:30:00,C30,0,0.0,0.0,N,5


,id,fecha,tipo_elem,intensidad,ocupacion,vmed,error,periodo_integracion
975194,6899,2023-01-01 00:00:00,C30,1140,4.0,69.0,N,5
975195,6899,2023-01-01 00:15:00,C30,1140,4.0,69.0,N,5
975196,6899,2023-01-01 00:30:00,C30,60,NaN,72.0,N,5
975197,6899,2023-01-01 00:45:00,C30,60,NaN,72.0,N,5
975198,6899,2023-01-01 01:00:00,C30,60,NaN,72.0,N,5
...,...,...,...,...,...,...,...,...
43123815,6899,2025-12-31 22:45:00,C30,0,0.0,0.0,N,5
43123816,6899,2025-12-31 23:00:00,C30,0,0.0,0.0,N,5
43123817,6899,2025-12-31 23:15:00,C30,0,0.0,0.0,N,5
43123818,6899,2025-12-31 23:30:00,C30,0,0.0,0.0,N,5


In [180]:
filtered_df = df[df['id'] == 1001]
filtered_df

counts = df['id'].value_counts()
counts

id
6678     110581
6685     110581
6699     110581
6700     110581
6742     110581
          ...  
11493      2544
11499      2469
11505      2469
11506      2469
11380       112
Name: count, Length: 464, dtype: int64

In [181]:
df = df[df['tipo_elem'].isin(['C30', 'M30'])]
print(f'Shape after filtering: {df.shape}')
df['tipo_elem'].value_counts()

Shape after filtering: (43409606, 8)


tipo_elem
M30    31061996
C30    12347610
Name: count, dtype: int64

In [182]:
df.dtypes

id                       int64
fecha                      str
tipo_elem                  str
intensidad               int64
ocupacion              float64
vmed                   float64
error                      str
periodo_integracion      int64
dtype: object

In [183]:
df.describe()

,id,intensidad,ocupacion,vmed,periodo_integracion
count,4.340961e+07,4.340961e+07,4.299259e+07,4.302552e+07,4.340961e+07
mean,6.125720e+03,1.169980e+03,5.536140e+00,6.163760e+01,1.219913e+01
std,2.425190e+03,1.269771e+03,8.895773e+00,2.520106e+01,4.482414e+00
min,1.001000e+03,0.000000e+00,0.000000e+00,-1.270000e+02,1.000000e+00
25%,6.640000e+03,2.050000e+02,1.000000e+00,5.100000e+01,5.000000e+00
50%,6.750000e+03,7.000000e+02,3.000000e+00,6.700000e+01,1.500000e+01
75%,6.880000e+03,1.761000e+03,7.000000e+00,8.000000e+01,1.500000e+01
max,1.150600e+04,2.502000e+04,1.000000e+02,2.460000e+02,3.000000e+01


In [184]:
df.isnull().sum()

id                           0
fecha                        0
tipo_elem                    0
intensidad                   0
ocupacion               417017
vmed                    384089
error                  1254638
periodo_integracion          0
dtype: int64

## 2. Inspect Errors and Missing Data

In [185]:
df[df['error'] != 'N'].sample(10)

,id,fecha,tipo_elem,intensidad,ocupacion,vmed,error,periodo_integracion
22884732,10264,2025-06-21 21:45:00,M30,0,0.0,0.0,NaN,15
10651020,6658,2024-03-13 19:30:00,M30,0,0.0,0.0,NaN,15
8896703,6937,2026-02-21 15:00:00,M30,0,0.0,0.0,NaN,15
29152253,6659,2025-08-22 13:45:00,M30,0,0.0,0.0,NaN,15
11865304,6683,2025-03-23 20:15:00,M30,0,0.0,0.0,NaN,15
9407427,3827,2023-03-20 06:00:00,M30,0,0.0,0.0,NaN,15
34599800,10662,2023-10-29 02:30:00,M30,45,0.0,34.0,NaN,30
21037356,6656,2024-06-22 01:00:00,M30,0,0.0,0.0,NaN,15
10649022,6657,2024-03-23 23:15:00,M30,0,0.0,0.0,NaN,15
8329604,6659,2026-02-24 01:00:00,M30,0,0.0,0.0,NaN,15


In [186]:
df.head(500)

,id,fecha,tipo_elem,intensidad,ocupacion,vmed,error,periodo_integracion
0,1001,2023-01-01 00:00:00,C30,0,0.0,0.0,N,5
1,1001,2023-01-01 00:15:00,C30,0,0.0,0.0,N,5
2,1001,2023-01-01 00:30:00,C30,180,0.0,66.0,N,5
3,1001,2023-01-01 00:45:00,C30,180,0.0,66.0,N,5
4,1001,2023-01-01 01:00:00,C30,180,0.0,66.0,N,5
...,...,...,...,...,...,...,...,...
495,1001,2023-01-06 03:45:00,C30,240,0.0,65.0,N,5
496,1001,2023-01-06 04:00:00,C30,216,0.0,58.0,N,5
497,1001,2023-01-06 04:15:00,C30,192,0.0,55.0,N,5
498,1001,2023-01-06 04:30:00,C30,132,NaN,60.0,N,5


In [187]:
df.isnull().sum()

id                           0
fecha                        0
tipo_elem                    0
intensidad                   0
ocupacion               417017
vmed                    384089
error                  1254638
periodo_integracion          0
dtype: int64

In [188]:
df['vmed'].value_counts(dropna=False)

vmed
0.0      2835589
66.0      937767
65.0      934712
64.0      926960
67.0      917287
          ...   
206.0          1
192.0          1
173.0          1
178.0          1
235.0          1
Name: count, Length: 329, dtype: int64

In [189]:
df['error'].value_counts(dropna=False)

error
N      42154968
NaN     1254638
Name: count, dtype: int64

In [190]:
df['vmed'].value_counts(dropna=False).sort_index()

vmed
-127.0      7969
-119.0        41
-118.0        33
-117.0        33
-116.0       174
           ...  
 232.0        13
 235.0         1
 237.0         1
 246.0         3
 NaN      384089
Name: count, Length: 329, dtype: int64

## 3. Data Processing: 15-min to Hourly Aggregation

### 3.1. Data cleaning
- Rows with `error='S'` or `error=NaN` are treated as invalid (all numeric fields set to NaN)
- Negative values in numeric fields indicate missing data and are set to NaN
- `intensidad` is kept in its original unit (vehicles/hour)

### 3.2. Hourly aggregation rules (per id, per hour)
- **4/4 valid quarters**: aggregate normally
- **3/4 valid quarters**: aggregate + `hora_baja_confianza=1`
- **<=2/4 valid quarters**: all values set to NaN + `hora_incompleta=1`

| Field | Aggregation |
|---|---|
| `intensidad` | mean (if >=3 valid) |
| `ocupacion` | mean (if >=3 valid) |
| `vmed` | mean (if >=3 valid) |
| `error` | worst status (S > E > N) |
| `periodo_integracion` | sum |

### Quality flags
| Flag | Description |
|---|---|
| `n_validos_15m` | Count of valid 15-min quarters (0-4) |
| `n_error_E` | Count of quarters with error='E' |
| `n_error_S` | Count of quarters with error='S' |
| `n_error_NaN` | Count of quarters with error=NaN (broken sensors) |
| `hora_incompleta` | 1 if n_validos <= 2 |
| `hora_baja_confianza` | 1 if n_validos == 3 |

In [191]:
# Invalidate rows with error='S' or NaN error (broken/erroneous sensors)
invalid_error = df['error'].isin(['S']) | df['error'].isna()
print(f'Rows with error=S or NaN: {invalid_error.sum()} ({invalid_error.sum() / len(df) * 100:.2f}%)')
for col in ['intensidad', 'ocupacion', 'vmed']:
    df.loc[invalid_error, col] = np.nan

# Replace negative values with NaN (negative = missing data per documentation)
for col in ['intensidad', 'ocupacion', 'vmed']:
    df[col] = df[col].where(df[col] >= 0, np.nan)

# Mark valid quarters (intensidad not NaN)
df['valido'] = df['intensidad'].notna().astype(int)
print(f'Valid quarters: {df["valido"].sum()} / {len(df)} ({df["valido"].mean() * 100:.1f}%)')

print(f'\nShape before aggregation: {df.shape}')
df.head()

Rows with error=S or NaN: 1254638 (2.89%)
Valid quarters: 42154968 / 43409606 (97.1%)

Shape before aggregation: (43409606, 9)


,id,fecha,tipo_elem,intensidad,ocupacion,vmed,error,periodo_integracion,valido
0,1001,2023-01-01 00:00:00,C30,0.0,0.0,0.0,N,5,1
1,1001,2023-01-01 00:15:00,C30,0.0,0.0,0.0,N,5,1
2,1001,2023-01-01 00:30:00,C30,180.0,0.0,66.0,N,5,1
3,1001,2023-01-01 00:45:00,C30,180.0,0.0,66.0,N,5,1
4,1001,2023-01-01 01:00:00,C30,180.0,0.0,66.0,N,5,1


### 3.2. Hourly aggregation

In [192]:
import pandas as pd

# Parse fecha and truncate to hour
df['fecha'] = pd.to_datetime(df['fecha'], format='%Y-%m-%d %H:%M:%S')
df['hora'] = df['fecha'].dt.floor('h')

# Create dummy columns for vectorized error counting
df['is_error_E'] = (df['error'] == 'E').astype('int8')
df['is_error_S'] = (df['error'] == 'S').astype('int8')
df['is_error_NaN'] = df['error'].isna().astype('int8')

print('Grouping...')

# Fully vectorized aggregation (sort=False avoids expensive sort on 4M+ groups)
group = df.groupby(['id', 'hora', 'tipo_elem'], sort=False, observed=True)
hourly_df = group.agg(
    intensidad=('intensidad', 'mean'),
    ocupacion=('ocupacion', 'mean'),
    vmed=('vmed', 'mean'),
    periodo_integracion=('periodo_integracion', 'sum'),
    n_validos_15m=('valido', 'sum'),
    n_error_E=('is_error_E', 'sum'),
    n_error_S=('is_error_S', 'sum'),
    n_error_NaN=('is_error_NaN', 'sum'),
    has_S=('is_error_S', 'max'),
    has_E=('is_error_E', 'max'),
).reset_index()

print('Computing flags...')

# Worst error per hour (vectorized)
hourly_df['error'] = np.where(hourly_df['has_S'] > 0, 'S',
                     np.where(hourly_df['has_E'] > 0, 'E', 'N'))
hourly_df.drop(columns=['has_S', 'has_E'], inplace=True)

# Quality flags
hourly_df['hora_incompleta'] = (hourly_df['n_validos_15m'] <= 2).astype(int)
hourly_df['hora_baja_confianza'] = (hourly_df['n_validos_15m'] == 3).astype(int)

# Set values to NaN for incomplete hours (<=2 valid quarters)
incomplete = hourly_df['hora_incompleta'] == 1
for col in ['intensidad', 'ocupacion', 'vmed']:
    hourly_df.loc[incomplete, col] = np.nan

hourly_df = hourly_df.rename(columns={'hora': 'fecha'})

# Sort by id and fecha for clean output
hourly_df = hourly_df.sort_values(['id', 'fecha']).reset_index(drop=True)

# Cast flag columns to int
for col in ['n_validos_15m', 'n_error_E', 'n_error_S', 'n_error_NaN', 'hora_incompleta', 'hora_baja_confianza']:
    hourly_df[col] = hourly_df[col].astype(int)

# Clean up temp columns
df.drop(columns=['is_error_E', 'is_error_S', 'is_error_NaN'], inplace=True)

print(f'Original 15-min rows: {len(df)}')
print(f'Hourly rows: {len(hourly_df)}')
print(f'Reduction ratio: {len(df) / len(hourly_df):.1f}x')
hourly_df.head(10)

Grouping...
Computing flags...
Original 15-min rows: 43409606
Hourly rows: 10931514
Reduction ratio: 4.0x


,id,fecha,tipo_elem,intensidad,ocupacion,vmed,periodo_integracion,n_validos_15m,n_error_E,n_error_S,n_error_NaN,error,hora_incompleta,hora_baja_confianza
0,1001,2023-01-01 00:00:00,C30,90.0,0.0,33.0,20,4,0,0,0,N,0,0
1,1001,2023-01-01 01:00:00,C30,180.0,0.0,66.0,20,4,0,0,0,N,0,0
2,1001,2023-01-01 02:00:00,C30,180.0,0.0,66.0,20,4,0,0,0,N,0,0
3,1001,2023-01-01 03:00:00,C30,180.0,0.0,66.0,20,4,0,0,0,N,0,0
4,1001,2023-01-01 04:00:00,C30,180.0,0.0,66.0,20,4,0,0,0,N,0,0
5,1001,2023-01-01 05:00:00,C30,180.0,0.0,66.0,20,4,0,0,0,N,0,0
6,1001,2023-01-01 06:00:00,C30,180.0,0.0,66.0,20,4,0,0,0,N,0,0
7,1001,2023-01-01 07:00:00,C30,180.0,0.0,66.0,20,4,0,0,0,N,0,0
8,1001,2023-01-01 08:00:00,C30,180.0,0.0,66.0,20,4,0,0,0,N,0,0
9,1001,2023-01-01 09:00:00,C30,180.0,0.0,66.0,20,4,0,0,0,N,0,0


In [193]:
# Quality flag distribution
print('=== Valid quarters per hour ===')
print(hourly_df['n_validos_15m'].value_counts().sort_index())
print(f'\n=== Incomplete hours (<=2 valid) ===')
print(hourly_df['hora_incompleta'].value_counts())
print(f'\n=== Low confidence hours (3 valid) ===')
print(hourly_df['hora_baja_confianza'].value_counts())
print(f'\n=== Error E quarters ===')
print(hourly_df['n_error_E'].value_counts().sort_index())
print(f'\n=== Error S quarters ===')
print(hourly_df['n_error_S'].value_counts().sort_index())

=== Valid quarters per hour ===
n_validos_15m
0      290242
1       54485
2       65538
3      115589
4    10405660
Name: count, dtype: int64

=== Incomplete hours (<=2 valid) ===
hora_incompleta
0    10521249
1      410265
Name: count, dtype: int64

=== Low confidence hours (3 valid) ===
hora_baja_confianza
0    10815925
1      115589
Name: count, dtype: int64

=== Error E quarters ===
n_error_E
0    10931514
Name: count, dtype: int64

=== Error S quarters ===
n_error_S
0    10931514
Name: count, dtype: int64


### 3.3. Final Hourly Dataset

In [194]:
# Use hourly_df as the working dataframe going forward
df = hourly_df
print(f'Final dataset shape: {df.shape}')
df.dtypes

Final dataset shape: (10931514, 14)


id                              int64
fecha                  datetime64[us]
tipo_elem                         str
intensidad                    float64
ocupacion                     float64
vmed                          float64
periodo_integracion             int64
n_validos_15m                   int64
n_error_E                       int64
n_error_S                       int64
n_error_NaN                     int64
error                             str
hora_incompleta                 int64
hora_baja_confianza             int64
dtype: object

In [195]:
df.isnull().sum()

id                          0
fecha                       0
tipo_elem                   0
intensidad             410265
ocupacion              438082
vmed                   500805
periodo_integracion         0
n_validos_15m               0
n_error_E                   0
n_error_S                   0
n_error_NaN                 0
error                       0
hora_incompleta             0
hora_baja_confianza         0
dtype: int64

In [196]:
df.head(100)

,id,fecha,tipo_elem,intensidad,ocupacion,vmed,periodo_integracion,n_validos_15m,n_error_E,n_error_S,n_error_NaN,error,hora_incompleta,hora_baja_confianza
0,1001,2023-01-01 00:00:00,C30,90.0,0.0,33.0,20,4,0,0,0,N,0,0
1,1001,2023-01-01 01:00:00,C30,180.0,0.0,66.0,20,4,0,0,0,N,0,0
2,1001,2023-01-01 02:00:00,C30,180.0,0.0,66.0,20,4,0,0,0,N,0,0
3,1001,2023-01-01 03:00:00,C30,180.0,0.0,66.0,20,4,0,0,0,N,0,0
4,1001,2023-01-01 04:00:00,C30,180.0,0.0,66.0,20,4,0,0,0,N,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,1001,2023-01-04 23:00:00,C30,300.0,1.0,56.0,20,4,0,0,0,N,0,0
96,1001,2023-01-05 00:00:00,C30,300.0,1.0,56.0,20,4,0,0,0,N,0,0
97,1001,2023-01-05 01:00:00,C30,300.0,1.0,56.0,20,4,0,0,0,N,0,0
98,1001,2023-01-05 02:00:00,C30,300.0,1.0,56.0,20,4,0,0,0,N,0,0


In [197]:
filtered_df = df[df['id'] == 1001]
filtered_df


,id,fecha,tipo_elem,intensidad,ocupacion,vmed,periodo_integracion,n_validos_15m,n_error_E,n_error_S,n_error_NaN,error,hora_incompleta,hora_baja_confianza
0,1001,2023-01-01 00:00:00,C30,90.0,0.0,33.0,20,4,0,0,0,N,0,0
1,1001,2023-01-01 01:00:00,C30,180.0,0.0,66.0,20,4,0,0,0,N,0,0
2,1001,2023-01-01 02:00:00,C30,180.0,0.0,66.0,20,4,0,0,0,N,0,0
3,1001,2023-01-01 03:00:00,C30,180.0,0.0,66.0,20,4,0,0,0,N,0,0
4,1001,2023-01-01 04:00:00,C30,180.0,0.0,66.0,20,4,0,0,0,N,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15342,1001,2024-10-01 07:00:00,C30,4560.0,12.0,57.0,20,4,0,0,0,N,0,0
15343,1001,2024-10-01 08:00:00,C30,4560.0,12.0,57.0,20,4,0,0,0,N,0,0
15344,1001,2024-10-01 09:00:00,C30,NaN,NaN,NaN,8,2,0,0,0,N,1,0
15345,1001,2025-04-01 09:00:00,C30,NaN,NaN,NaN,3,1,0,0,0,N,1,0


In [198]:
total_unique_ids = df['id'].nunique()
print(f"Total unique ids: {total_unique_ids}")

Total unique ids: 464


In [199]:
counts = df['id'].value_counts()
counts

id
3489     27662
3531     27662
3533     27662
3539     27662
3560     27662
         ...  
11493      636
11499      618
11505      618
11506      618
11380       49
Name: count, Length: 464, dtype: int64

In [200]:
# Step 2: Find what the maximum count is
max_count = counts.max()

# Step 3: Count how many IDs share that maximum count
# (counts == max_count) creates True/False values. .sum() counts the True values.
number_of_max_ids = (counts == max_count).sum()

print(f"Max count is: {max_count}")
print(f"Number of IDs with this max count: {number_of_max_ids}")

Max count is: 27662
Number of IDs with this max count: 93


In [201]:
import pandas as pd

# df_hourly = your hourly-aggregated dataframe
# assumes one row per (id, hour)

id_counts = (
    df.groupby("id")
    .size()
    .reset_index(name="n_rows")
    .sort_values("n_rows", ascending=False)
    .reset_index(drop=True)
)

total_rows = id_counts["n_rows"].sum()

id_counts["pct_of_total_rows"] = 100 * id_counts["n_rows"] / total_rows
id_counts["cum_rows"] = id_counts["n_rows"].cumsum()
id_counts["cum_pct_of_total_rows"] = 100 * id_counts["cum_rows"] / total_rows
id_counts["rank"] = id_counts.index + 1

print(id_counts.head(20))

       id  n_rows  pct_of_total_rows  cum_rows  cum_pct_of_total_rows  rank
0    6739   27662           0.253048     27662               0.253048     1
1    6757   27662           0.253048     55324               0.506096     2
2    6684   27662           0.253048     82986               0.759145     3
3    3560   27662           0.253048    110648               1.012193     4
4    6685   27662           0.253048    138310               1.265241     5
5    6686   27662           0.253048    165972               1.518289     6
6    6699   27662           0.253048    193634               1.771337     7
7    6700   27662           0.253048    221296               2.024386     8
8    6702   27662           0.253048    248958               2.277434     9
9   10661   27662           0.253048    276620               2.530482    10
10   3539   27662           0.253048    304282               2.783530    11
11   6713   27662           0.253048    331944               3.036578    12
12   6716   

In [202]:
thresholds = [24, 48, 96, 168, 336, 720, 1000, 2000, 3000, 5000, 10000, 20000, 22000, 25000, 26000, 27000]

summary = []
for t in thresholds:
    kept = id_counts[id_counts["n_rows"] >= t]
    n_ids = len(kept)
    n_rows_kept = kept["n_rows"].sum()
    pct_rows_kept = 100 * n_rows_kept / total_rows if total_rows > 0 else 0

    summary.append({
        "min_rows_threshold": t,
        "n_ids_kept": n_ids,
        "pct_ids_kept": 100 * n_ids / len(id_counts),
        "rows_kept": n_rows_kept,
        "pct_total_rows_kept": pct_rows_kept,
    })

threshold_summary = pd.DataFrame(summary)
threshold_summary

,min_rows_threshold,n_ids_kept,pct_ids_kept,rows_kept,pct_total_rows_kept
0,24,464,100.000000,10931514,100.000000
1,48,464,100.000000,10931514,100.000000
2,96,463,99.784483,10931465,99.999552
3,168,463,99.784483,10931465,99.999552
4,336,463,99.784483,10931465,99.999552
5,720,459,98.922414,10928975,99.976774
6,1000,442,95.258621,10915753,99.855821
7,2000,439,94.612069,10912743,99.828285
8,3000,439,94.612069,10912743,99.828285
9,5000,434,93.534483,10888579,99.607236


## 4. Filter by Minimum Coverage

Keep only measurement points with at least 25,000 hourly records to ensure sufficient temporal coverage for analysis.

In [203]:
MIN_HOURLY_RECORDS = 25_000

counts = df['id'].value_counts()
valid_ids = counts[counts >= MIN_HOURLY_RECORDS].index
n_before = df['id'].nunique()

df = df[df['id'].isin(valid_ids)]

print(f'IDs before: {n_before} → after: {df["id"].nunique()}')
print(f'Rows before: {counts.sum():,} → after: {len(df):,} ({len(df)/counts.sum()*100:.1f}%)')
df.shape

IDs before: 464 → after: 351
Rows before: 10,931,514 → after: 9,497,014 (86.9%)


(9497014, 14)

In [204]:
df.isnull().sum()

id                          0
fecha                       0
tipo_elem                   0
intensidad             361892
ocupacion              382884
vmed                   436154
periodo_integracion         0
n_validos_15m               0
n_error_E                   0
n_error_S                   0
n_error_NaN                 0
error                       0
hora_incompleta             0
hora_baja_confianza         0
dtype: int64

In [205]:
df.columns

Index(['id', 'fecha', 'tipo_elem', 'intensidad', 'ocupacion', 'vmed',
       'periodo_integracion', 'n_validos_15m', 'n_error_E', 'n_error_S',
       'n_error_NaN', 'error', 'hora_incompleta', 'hora_baja_confianza'],
      dtype='str')

### Reindex to complete hourly range

Before imputation, reindex each sensor to the full hourly range (2023-01-01 00:00 to 2026-02-28 23:00 = 27,720 hours). This ensures:
- Missing hours become explicit NaN rows instead of being silently absent.
- `interpolate(limit=3)` correctly counts actual time gaps (not row positions).
- Seasonal fallbacks can fill globally missing hours using historical patterns.
- All sensors end up with exactly 27,720 rows.

In [206]:
full_range = pd.date_range('2023-01-01', '2026-02-28 23:00:00', freq='h')
print(f'Full hourly range: {len(full_range)} hours')
print(f'Before reindex: {len(df)} rows, {df["id"].nunique()} sensors')

# Reindex each sensor to the full hourly range
# tipo_elem is constant per id, so we forward-fill it after reindex
dfs = []
for sensor_id, group in df.groupby('id'):
    tipo = group['tipo_elem'].iloc[0]
    group = group.set_index('fecha').reindex(full_range).rename_axis('fecha').reset_index()
    group['id'] = sensor_id
    group['tipo_elem'] = tipo
    dfs.append(group)

df = pd.concat(dfs, ignore_index=True)
df = df.sort_values(['id', 'fecha']).reset_index(drop=True)

# Fill quality flag columns for new rows (no data = incomplete hour)
for col in ['n_validos_15m', 'n_error_E', 'n_error_S', 'n_error_NaN']:
    df[col] = df[col].fillna(0).astype(int)
df['hora_incompleta'] = df['hora_incompleta'].fillna(1).astype(int)
df['hora_baja_confianza'] = df['hora_baja_confianza'].fillna(0).astype(int)
df['error'] = df['error'].fillna('N')
df['periodo_integracion'] = df['periodo_integracion'].fillna(0)

print(f'After reindex: {len(df)} rows')
print(f'Expected: {len(full_range)} hours x {df["id"].nunique()} sensors = {len(full_range) * df["id"].nunique()}')
print(f'Rows per sensor: {df.groupby("id").size().unique()}')

Full hourly range: 27720 hours
Before reindex: 9497014 rows, 351 sensors
After reindex: 9729720 rows
Expected: 27720 hours x 351 sensors = 9729720
Rows per sensor: [27720]


In [207]:
# Definimos las columnas a imputar
cols_to_impute = ['intensidad', 'ocupacion', 'vmed']

print("Nulos antes de imputar:")
print(df[cols_to_impute].isnull().sum())

# === 0. CREAR LAS MÁSCARAS Y EL PESO ANTES DE CUALQUIER IMPUTACIÓN ===
df['intensidad_is_imputed'] = df['intensidad'].isna().astype(str)
df['ocupacion_is_imputed'] = df['ocupacion'].isna().astype(str)
df['vmed_is_imputed'] = df['vmed'].isna().astype(str)

# 1.0 para datos reales (no nulos), 0.0 para datos que vamos a imputar
df['peso_loss'] = df['intensidad'].notna().astype(float)
# =====================================================================

# PASO 1: Interpolación Lineal para huecos cortos (límite de 3 horas consecutivas)
# Ahora que el índice temporal es completo, limit=3 cuenta horas reales
for col in cols_to_impute:
    df[col] = df.groupby('id')[col].transform(
        lambda x: x.interpolate(method='linear', limit=3)
    )

# Extraemos las componentes temporales
df['mes'] = df['fecha'].dt.month
df['dia_semana'] = df['fecha'].dt.dayofweek
df['hora_dia'] = df['fecha'].dt.hour

# PASO 2: Imputación Estacional Estricta (Mes + Día Semana + Hora)
means_strict = df.groupby(['id', 'mes', 'dia_semana', 'hora_dia'])[cols_to_impute].transform('mean')

for col in cols_to_impute:
    df[col] = df[col].fillna(means_strict[col])

nulls_pending = df.isnull().sum()
print(f"Nulos pendientes después de imputación estacional estricta:\n{nulls_pending}")

# PASO 3: Fallback 1: Imputación Estacional Relajada (Día Semana + Hora)
if df[cols_to_impute].isnull().any().any():
    print("*** WARNING! - FALLBACK 1 ***")
    means_relaxed = df.groupby(['id', 'dia_semana', 'hora_dia'])[cols_to_impute].transform('mean')
    for col in cols_to_impute:
        df[col] = df[col].fillna(means_relaxed[col])

# PASO 4: Fallback 2: Media de la hora general
if df[cols_to_impute].isnull().any().any():
    print("*** WARNING! - FALLBACK 2 ***")
    means_fallback = df.groupby(['id', 'hora_dia'])[cols_to_impute].transform('mean')
    for col in cols_to_impute:
        df[col] = df[col].fillna(means_fallback[col])

# Limpieza de las columnas temporales auxiliares
df = df.drop(columns=['mes', 'dia_semana', 'hora_dia'])

Nulos antes de imputar:
intensidad    594598
ocupacion     615590
vmed          668860
dtype: int64
Nulos pendientes después de imputación estacional estricta:
fecha                        0
id                           0
tipo_elem                    0
intensidad               26524
ocupacion                26524
vmed                     26524
periodo_integracion          0
n_validos_15m                0
n_error_E                    0
n_error_S                    0
n_error_NaN                  0
error                        0
hora_incompleta              0
hora_baja_confianza          0
intensidad_is_imputed        0
ocupacion_is_imputed         0
vmed_is_imputed              0
peso_loss                    0
mes                          0
dia_semana                   0
hora_dia                     0
dtype: int64
*** WARNING! - FALLBACK 1 ***


In [208]:
df.isnull().sum()

fecha                    0
id                       0
tipo_elem                0
intensidad               0
ocupacion                0
vmed                     0
periodo_integracion      0
n_validos_15m            0
n_error_E                0
n_error_S                0
n_error_NaN              0
error                    0
hora_incompleta          0
hora_baja_confianza      0
intensidad_is_imputed    0
ocupacion_is_imputed     0
vmed_is_imputed          0
peso_loss                0
dtype: int64

In [209]:
df.columns

Index(['fecha', 'id', 'tipo_elem', 'intensidad', 'ocupacion', 'vmed',
       'periodo_integracion', 'n_validos_15m', 'n_error_E', 'n_error_S',
       'n_error_NaN', 'error', 'hora_incompleta', 'hora_baja_confianza',
       'intensidad_is_imputed', 'ocupacion_is_imputed', 'vmed_is_imputed',
       'peso_loss'],
      dtype='str')

In [210]:
counts = df['id'].value_counts()
counts

id
1006     27720
1011     27720
1017     27720
1019     27720
1020     27720
         ...  
10268    27720
10659    27720
10660    27720
10661    27720
10662    27720
Name: count, Length: 351, dtype: int64

In [211]:
df["peso_loss"].value_counts()

peso_loss
1.0    9135122
0.0     594598
Name: count, dtype: int64

## 7. Feature enrichment

In [212]:
# Temporal features extracted from fecha
df.insert(df.columns.get_loc('fecha') + 1, 'dia_semana', df['fecha'].dt.dayofweek)   # 0=Monday, 6=Sunday
df.insert(df.columns.get_loc('dia_semana') + 1, 'hora_dia', df['fecha'].dt.hour)     # 0-23
df.insert(df.columns.get_loc('hora_dia') + 1, 'mes', df['fecha'].dt.month)           # 1-12
df[['fecha', 'dia_semana', 'hora_dia', 'mes']].head()

,fecha,dia_semana,hora_dia,mes
0,2023-01-01 00:00:00,6,0,1
1,2023-01-01 01:00:00,6,1,1
2,2023-01-01 02:00:00,6,2,1
3,2023-01-01 03:00:00,6,3,1
4,2023-01-01 04:00:00,6,4,1


In [213]:
import holidays

# National + Comunidad de Madrid holidays (actual dates, not observed substitutes)
madrid_holidays = holidays.Spain(prov='MD', years=range(2023, 2027), observed=False)

# Add city-level holidays: San Isidro (May 15) and La Almudena (Nov 9)
for year in range(2023, 2027):
    madrid_holidays[pd.Timestamp(year, 5, 15)] = 'San Isidro'
    madrid_holidays[pd.Timestamp(year, 11, 9)] = 'La Almudena'

df.insert(
    df.columns.get_loc('mes') + 1,
    'festivo',
    df['fecha'].dt.date.isin(madrid_holidays).astype(int)
)

print(f'Holiday hours: {df["festivo"].sum():,} / {len(df):,}')
print(f'Unique holiday dates: {df[df["festivo"] == 1]["fecha"].dt.date.nunique()}')
df[df['festivo'] == 1][['fecha', 'dia_semana', 'festivo']].drop_duplicates(subset='fecha').head(20)

Holiday hours: 395,928 / 9,729,720
Unique holiday dates: 47


,fecha,dia_semana,festivo
0,2023-01-01 00:00:00,6,1
1,2023-01-01 01:00:00,6,1
2,2023-01-01 02:00:00,6,1
3,2023-01-01 03:00:00,6,1
4,2023-01-01 04:00:00,6,1
5,2023-01-01 05:00:00,6,1
6,2023-01-01 06:00:00,6,1
7,2023-01-01 07:00:00,6,1
8,2023-01-01 08:00:00,6,1
9,2023-01-01 09:00:00,6,1


In [214]:
# Eve of holiday: current day is not festivo but the next day is
next_day_festivo = df['fecha'].apply(
    lambda x: (x + pd.Timedelta(days=1)).date() in madrid_holidays
).astype(int)

df.insert(
    df.columns.get_loc('festivo') + 1,
    'vispera_festivo',
    ((df['festivo'] == 0) & (next_day_festivo == 1)).astype(int)
)

print(f'Vispera hours: {df["vispera_festivo"].sum():,} / {len(df):,}')
df[df['vispera_festivo'] == 1][['fecha', 'dia_semana', 'festivo', 'vispera_festivo']].drop_duplicates(subset='fecha').head(20)

Vispera hours: 336,960 / 9,729,720


,fecha,dia_semana,festivo,vispera_festivo
96,2023-01-05 00:00:00,3,0,1
97,2023-01-05 01:00:00,3,0,1
98,2023-01-05 02:00:00,3,0,1
99,2023-01-05 03:00:00,3,0,1
100,2023-01-05 04:00:00,3,0,1
101,2023-01-05 05:00:00,3,0,1
102,2023-01-05 06:00:00,3,0,1
103,2023-01-05 07:00:00,3,0,1
104,2023-01-05 08:00:00,3,0,1
105,2023-01-05 09:00:00,3,0,1


## Saving to CSV

In [215]:
Path('dataset/processed').mkdir(parents=True, exist_ok=True)
df.to_csv('dataset/processed/traffic.csv', index=False)
print(f'Saved {len(df):,} rows to dataset/processed/traffic.csv')

Saved 9,729,720 rows to dataset/processed/traffic.csv
